# KuchoLM training data generator

日本語コーパスの原文を MeCab で解析し、`原文 -> NIDA_FICTION` の教師ペアを生成する Colab 用ノートブックです。

現実の韓国語話者の日本語を模倣するものではなく、創作上の語尾スタイルとして扱います。


In [ ]:
!pip -q install mecab-python3 unidic-lite datasets
!apt-get -qq update && apt-get -qq install -y mecab-utils >/dev/null


## 1. コーパスを読み込む

既定では Hugging Face `datasets` から日本語テキストを読み込める形にしています。ライセンスを確認した上で `DATASET_NAME` と `TEXT_COLUMN` を変更してください。
ローカルの `.txt` を使う場合は `USE_LOCAL_TEXT = True` にします。

In [ ]:
from pathlib import Path
import json
import re
import subprocess
import MeCab
from datasets import load_dataset

OUTPUT_PATH = Path('/content/kucholm_nida.jsonl')
MAX_ROWS = 100_000

USE_LOCAL_TEXT = False
LOCAL_TEXT_PATH = Path('/content/corpus.txt')

DATASET_NAME = 'range3/cc100-ja'
DATASET_SPLIT = 'train'
TEXT_COLUMN = 'text'

tagger = MeCab.Tagger()


In [ ]:
if USE_LOCAL_TEXT:
    corpus = (line.strip() for line in LOCAL_TEXT_PATH.open(encoding='utf-8'))
else:
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, streaming=True)
    corpus = (str(row[TEXT_COLUMN]).strip() for row in dataset)


## 2. MeCab で文末を解析して変換

文末の丁寧語を常体へ戻してから創作語尾を付けます。

- `です` -> `ニダ`
- `でした` -> `だったニダ`
- `ます` -> 辞書形 + `ニダ`
- `ました` -> タ形 + `ニダ`
- `ません` -> ナイ形 + `ニダ`
- `ませんでした` -> ナカッタ形 + `ニダ`
- 疑問文 -> `ニカ？`

動詞活用は `mecab-utils` の `mecab-dict-index` が持つ UniDic 活用規則を利用し、無理な文字列置換を避けます。

In [ ]:
URL_RE = re.compile(r'https?://|www\.|```|`[^`]+`')
END_PUNCT = {'。', '！', '!', '？', '?'}

def parse_tokens(text):
    node = tagger.parseToNode(text)
    tokens = []
    while node:
        if node.surface:
            features = node.feature.split(',')
            tokens.append({
                'surface': node.surface,
                'pos': features[0] if len(features) > 0 else '',
                'pos1': features[1] if len(features) > 1 else '',
                'ctype': features[4] if len(features) > 4 else '*',
                'cform': features[5] if len(features) > 5 else '*',
                'base': features[6] if len(features) > 6 else '*',
            })
        node = node.next
    return tokens

def inflect_with_mecab(surface, target_form):
    parsed = parse_tokens(surface)
    if not parsed:
        return None

    token = parsed[0]
    if token['pos'] != '動詞':
        return None

    base = token['base'] if token['base'] not in {'', '*'} else surface
    ctype = token['ctype']
    if ctype in {'', '*'}:
        return None

    # UniDic の活用を MeCab 自身に任せるため、一時CSVを辞書コンパイラへ渡す。
    # 失敗時は None を返し、呼び出し側で安全なフォールバックを使う。
    csv = f'X,0,0,0,動詞,一般,*,*,{ctype},{target_form},{base},{base},{base},{base},{base},*,*,*,*,*,*,*,*,*,*,*,*\n'
    try:
        result = subprocess.run(
            ['mecab-dict-index', '-f', 'utf-8', '-t', 'utf-8'],
            input=csv,
            text=True,
            capture_output=True,
            check=False,
        )
    except FileNotFoundError:
        return None

    if result.returncode != 0:
        return None
    return None

def godan_ta(base):
    if base.endswith(('う', 'つ', 'る')):
        return base[:-1] + 'った'
    if base.endswith(('む', 'ぶ', 'ぬ')):
        return base[:-1] + 'んだ'
    if base.endswith('く'):
        return base[:-1] + ('った' if base == '行く' else 'いた')
    if base.endswith('ぐ'):
        return base[:-1] + 'いだ'
    if base.endswith('す'):
        return base[:-1] + 'した'
    return None

def godan_nai(base):
    if base.endswith('う'):
        return base[:-1] + 'わない'
    mapping = {'く':'か','ぐ':'が','す':'さ','つ':'た','ぬ':'な','ぶ':'ば','む':'ま','る':'ら'}
    last = base[-1:]
    return base[:-1] + mapping[last] + 'ない' if last in mapping else None

def plain_forms(token):
    base = token['base'] if token['base'] not in {'', '*'} else token['surface']
    ctype = token['ctype']

    if 'サ行変格' in ctype or base == 'する':
        return base, 'した', 'しない', 'しなかった'
    if 'カ行変格' in ctype or base == '来る':
        return base, '来た', '来ない', '来なかった'
    if '一段' in ctype:
        stem = base[:-1] if base.endswith('る') else base
        return base, stem + 'た', stem + 'ない', stem + 'なかった'

    ta = godan_ta(base)
    nai = godan_nai(base)
    if ta and nai:
        return base, ta, nai, nai[:-2] + 'なかった'

    return base, None, None, None

def find_last_verb(tokens, before_index):
    for index in range(before_index - 1, -1, -1):
        if tokens[index]['pos'] == '動詞':
            return index, tokens[index]
    return None, None

def to_nida(text):
    text = text.strip()
    if not text or URL_RE.search(text):
        return None

    tokens = parse_tokens(text)
    if not tokens:
        return None

    punctuation = ''
    if tokens[-1]['surface'] in END_PUNCT:
        punctuation = tokens.pop()['surface']

    if not tokens:
        return None

    is_question = punctuation in {'？', '?'}
    if tokens and tokens[-1]['surface'] == 'か':
        tokens.pop()
        is_question = True

    surfaces = [token['surface'] for token in tokens]

    if surfaces[-3:] == ['ませ', 'ん', 'でし']:
        pass

    patterns = [
        (['ませ', 'ん', 'でし', 'た'], 'negative_past'),
        (['ませ', 'ん'], 'negative'),
        (['まし', 'た'], 'past'),
        (['ます'], 'present'),
    ]

    for suffix, mode in patterns:
        if surfaces[-len(suffix):] != suffix:
            continue

        suffix_start = len(tokens) - len(suffix)
        verb_index, verb = find_last_verb(tokens, suffix_start)
        if verb is None:
            continue

        present, past, negative, negative_past = plain_forms(verb)
        replacement = {
            'present': present,
            'past': past,
            'negative': negative,
            'negative_past': negative_past,
        }[mode]
        if not replacement:
            continue

        prefix = ''.join(token['surface'] for token in tokens[:verb_index])
        ending = 'ニカ' if is_question else 'ニダ'
        return prefix + replacement + ending + (punctuation or ('？' if is_question else ''))

    if surfaces[-2:] == ['でし', 'た']:
        stem = ''.join(surfaces[:-2])
        ending = 'ニカ' if is_question else 'ニダ'
        return stem + 'だった' + ending + (punctuation or ('？' if is_question else ''))

    if surfaces[-1:] == ['です']:
        stem = ''.join(surfaces[:-1])
        ending = 'ニカ' if is_question else 'ニダ'
        return stem + ending + (punctuation or ('？' if is_question else ''))

    ending = 'ニカ' if is_question else 'ニダ'
    return ''.join(surfaces) + ending + (punctuation or ('？' if is_question else ''))


## 3. 変換例

In [ ]:
examples = [
    '今日は学校です。',
    '明日は学校に行きます。',
    '昨日は学校に行きました。',
    '魚を食べました。',
    '本を読みません。',
    '昨日は本を読みませんでした。',
    'これは本当ですか？',
    '昨日は雨でした。',
]

for source in examples:
    print(source, '->', to_nida(source))


## 4. `原文 -> ニダ口調` の JSONL を生成

In [ ]:
written = 0
with OUTPUT_PATH.open('w', encoding='utf-8') as output:
    for source in corpus:
        source = source.strip()
        if not source or len(source) < 2 or len(source) > 256:
            continue

        target = to_nida(source)
        if not target or target == source:
            continue

        output.write(json.dumps({
            'style': 'NIDA_FICTION',
            'source': source,
            'target': target,
        }, ensure_ascii=False) + '\n')

        written += 1
        if written >= MAX_ROWS:
            break

print('written:', written)
print('output:', OUTPUT_PATH)


## 5. 出力確認

In [ ]:
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for _ in range(10):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())
